# TP de clustering — Licence 3

Dans ce notebook, on applique les méthodes vues en cours : **K-Means** et **CAH**. On prépare les données, on regarde le coude, puis on compare les partitions avec le score de silhouette.

### Importer les librairies

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram
from yellowbrick.cluster import KElbowVisualizer

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data sets').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'data sets'

### Importer et lire les données

Chaque ligne représente une graine de blé décrite par sept mesures morphologiques. Le but est de regrouper les graines selon leurs mesures.

**Fichier :** `data sets/wheat_seeds_dataset.csv` (fourni avec le cours).

In [ ]:
df = pd.read_csv(DATA_DIR / 'wheat_seeds_dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().T

### Préparation des données

On vérifie les valeurs manquantes, puis on normalise les variables comme dans le TP. La normalisation met les observations à une échelle comparable.

In [ ]:
df.isnull().sum()
df = df.dropna().copy()
df.index = ['Graine' + str(i + 1) for i in range(len(df))]
x = df.values
x = normalize(x)
x[:5]

In [ ]:
pca_model = PCA(n_components=2)
features_pca = pca_model.fit_transform(x)
df_pca = pd.DataFrame(features_pca, columns=['PC1', 'PC2'])
round(pca_model.explained_variance_ratio_.sum() * 100, 2)

### K-Means

#### Choisir le nombre de groupes avec la méthode du coude

In [ ]:
kmeans_model = KMeans(init='random', max_iter=500, random_state=42)
Elbow_M = KElbowVisualizer(kmeans_model, k=(2, 10))
Elbow_M.fit(x)
Elbow_M.show()

#### Appliquer K-Means

On utilise 3 groupes, comme point de départ indiqué dans le TP.

In [ ]:
kmeans_model = KMeans(n_clusters=3, init='random', max_iter=500, random_state=42)
kmeans_model.fit(x)
kmeans_partition = kmeans_model.predict(x)
kmeans_partition[:20]

In [ ]:
val_kmeans = pd.Series(kmeans_partition).value_counts().sort_index()
val_kmeans

In [ ]:
fig = px.bar(x=val_kmeans.index.astype(str), y=val_kmeans.values,
             color=val_kmeans.index.astype(str), text_auto=True,
             labels={'x':'Groupe','y':'Nombre de graines'})
fig.show()

In [ ]:
fig = px.scatter(df_pca, x='PC1', y='PC2', color=kmeans_partition.astype(str),
                 labels={'color':'Groupe'}, title='Groupes obtenus avec K-Means')
fig.show()

In [ ]:
sil_kmeans = silhouette_score(x, kmeans_partition)
print('Score silhouette K-Means :', round(sil_kmeans, 3))

### CAH (Classification Ascendante Hiérarchique)

In [ ]:
Z = linkage(x, method='average')
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30)
plt.title('Dendrogramme de la CAH')
plt.xlabel('Observations ou groupes')
plt.ylabel('Distance')
plt.show()

In [ ]:
cah_model = AgglomerativeClustering(n_clusters=3, linkage='average')
cah_partition = cah_model.fit_predict(x)
cah_partition[:20]

In [ ]:
val_cah = pd.Series(cah_partition).value_counts().sort_index()
val_cah

In [ ]:
fig = px.bar(x=val_cah.index.astype(str), y=val_cah.values,
             color=val_cah.index.astype(str), text_auto=True,
             labels={'x':'Groupe','y':'Nombre de graines'})
fig.show()

In [ ]:
fig = px.scatter(df_pca, x='PC1', y='PC2', color=cah_partition.astype(str),
                 labels={'color':'Groupe'}, title='Groupes obtenus avec la CAH')
fig.show()

In [ ]:
sil_cah = silhouette_score(x, cah_partition)
print('Score silhouette CAH :', round(sil_cah, 3))

### Comparaison des deux méthodes

In [ ]:
Model_Names = ['KMEANS', 'CAH']
sil = [sil_kmeans, sil_cah]
fig = px.bar(x=Model_Names, y=sil, color=Model_Names, text_auto=True,
             labels={'x':'Méthode','y':'Score silhouette'},
             title='Comparaison des scores silhouette')
fig.show()

### Conclusion

On compare les deux scores silhouette pour voir quelle méthode sépare le mieux les observations. L'ACP sert seulement à afficher les groupes en deux dimensions ; les calculs de clustering utilisent les variables normalisées.